# Data Cleaning 02 -- FRED Daily

## Input
`Data/Data_Collection/Initial/02_FRED/fred_daily.parquet`

## Purpose
Cleans daily macro/financial data from the FRED API. Keyed on `date` only (no PERMNO). Key concerns addressed: weekend and holiday rows in the FRED calendar-day file, series with different start dates (e.g., SOFR began 2018), discontinued series (LIBOR ended 2023, TED spread ended ~2022), derived factors inheriting NaN from their inputs, and a duplicate FRED ID bug where `ism_prices` and `ppi_final` both mapped to the same series (PPIFIS).

## Stage 0: Load & Inspect
- Weekend rows (Saturday/Sunday) are dropped
- Holiday rows are dropped (rows where everything except `fed_funds_eff` is NaN)
- Basic shape, date range, and column inventory reported

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts sorted descending, with first/last valid dates and flags for columns above 30% or 50% NaN
- Per-row NaN distribution and identification of the worst rows

## Stage 2: Gap Analysis
- **Date frequency check:** confirms the file is on a trading-day calendar after weekend removal, reports gap distribution
- **Per-series coverage windows:** each series is classified as OK, LATE START (first valid date after 2010), DISCONTINUED (last valid date before 2023), or SPARSE (<50% valid)
- **Consecutive gap analysis:** for key series (`yield_10y`, `yield_2y`, `fed_funds_eff`, `hy_oas`, `ig_oas`, `wti_oil`, `gold`, `sp500`, `sofr`, `slope_2y10y`), identifies the longest consecutive NaN runs and flags any runs longer than 3 days for investigation

## Stage 3: Specific Issue Checks
- **Duplicate FRED ID bug:** confirms `ism_prices` and `ppi_final` are identical (both pulled from PPIFIS) via value equality and correlation check
- **Discontinued series:** lists all series whose last valid date falls before 2024
- **Value range sanity checks:** verifies key series fall within expected ranges (e.g., 10Y yield 0--10%, WTI oil -$50--$200, gold $200--$3000, S&P 500 500--7000)
- **Weekend/holiday row count** after initial filtering

## Stage 5: Clean & Save

### Columns Dropped (18)
- `margin_debt` -- quarterly data in a daily table (58 obs out of ~5,479).
- `breakeven_20y`, `breakeven_30y` -- monthly frequency and hasn't got enough data (177 and 129 obs respectively).
- `soma_mbs` -- discontinued June 2018, missing the entire 2020--2024 test period
- `stlfsi` -- St. Louis Fed Financial Stress Index discontinued January 2022, missing the last 3 years
- `ted_spread` -- discontinued January 2022 (LIBOR cessation), structurally missing for the entire 2022--2024 test window
- `mortgage_30y`, `mortgage_15y` -- weekly series (1,095 obs).
- `soma_treasury` -- weekly series (1,095 obs), same issue
- `nfci`, `anfci` -- weekly series (1,096 obs), same issue
- `sofr` -- only starts April 2018 (69% NaN), too much of the training period missing
- `iorb` -- only starts July 2021 (84% NaN), almost entirely missing
- `sp500`, `sp500_ret_1d`, `sp500_ret_5d` -- only starts March 2016 on FRED, redundant with CRSP target return and Fama-French `mktrf` which have full 2004--2024 coverage
- `rrp` -- reverse repo facility, 47% NaN, sparse reporting before becoming daily in later years
- `tips_30y` -- only starts February 2010 (32% NaN), too much training period missing

### Holiday Rows Dropped
Rows where >50% of remaining factors are NaN after column drops. These are US market holidays that fall on weekdays (e.g., New Year's Day, MLK Day, Good Friday, Thanksgiving, Christmas). FRED reports `fed_funds_eff` and `discount_rate` on these days but all market-based series are NaN.

### Forward-Fill (Limit 5 Trading Days)
Remaining NaN are isolated single-day gaps from minor calendar mismatches between different FRED source agencies (e.g., Treasury yields vs ICE credit spreads vs commodity exchanges may observe different holidays). A 5-day limit prevents filling across genuine data outages while covering all normal holiday gaps.

### Remaining NaN (Structural, Not Cleaned)
- `twexb`, `twexm` -- trade-weighted dollar indices start January 2006 (~500 leading NaN)
- `tips_20y` -- 20-year TIPS first issued mid-2004 (~141 leading NaN)
- These are structural: the instruments did not exist before those dates. Resolved when the merged dataset starts from 2006.

## Output
`Data/Data_Collection/Cleaned/02_FRED/fred_daily_clean.parquet` -- 50 factor columns (down from 68 before cleaning)

In [2]:
# %% [markdown]
# # Data Cleaning: fred_daily.parquet
#
# Source: Data/Data_Collection/Initial/02_FRED/fred_daily.parquet
# Output: Data/Data_Collection/Cleaned/02_FRED/fred_daily_clean.parquet
#
# This is daily macro/financial data from the FRED API. No PERMNO — keyed on
# date only. Key concerns:
#   - Some series have different start dates (e.g., SOFR began 2018)
#   - Some series are discontinued (LIBOR ended 2023, TED spread ended ~2022)
#   - Weekend/holiday rows may be present (FRED uses calendar dates)
#   - Derived factors (slopes, spreads, ratios) inherit NaN from their inputs
#   - The duplicate FRED ID bug: ism_prices = ppi_final (same series PPIFIS)

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../../../Data/Data_Collection/Initial/02_FRED/fred_daily.parquet')
OUT_DIR  = Path('../../../Data/Data_Collection/Cleaned/02_FRED')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — fred_daily")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])

# ── Drop weekends ────────────────────────────────────────────────────────────
n_before = len(df)
df = df[df['date'].dt.dayofweek < 5]
n_weekends = n_before - len(df)
print(f"\nDropped {n_weekends:,} weekend rows ({n_before:,} → {len(df):,})")

# ── Drop holidays (rows where everything except fed_funds_eff is NaN) ────────
factor_cols = [c for c in df.columns if c != 'date']
cols_ex_ff = [c for c in factor_cols if c != 'fed_funds_eff']

n_before = len(df)
all_nan_mask = df[cols_ex_ff].isna().all(axis=1)
df = df[~all_nan_mask]
n_holidays = n_before - len(df)
print(f"Dropped {n_holidays:,} holiday rows ({n_before:,} → {len(df):,})")

print(f"\nAfter filtering:")
print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique dates: {df['date'].nunique():,}")

factor_cols = [c for c in df.columns if c != 'date']

print(f"\nFactor columns ({len(factor_cols)}):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<35s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (5 rows, first 8 factors) ---")
print(df[['date'] + factor_cols[:8]].head(5).to_string(index=False))

print(f"\n--- Tail (5 rows, first 8 factors) ---")
print(df[['date'] + factor_cols[:8]].tail(5).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT — fred_daily")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN (sorted descending) ───────────────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN Summary ---")
print(f"  Factors with   0% NaN: {(col_nan_pct == 0).sum()}")
print(f"  Factors with  <5% NaN: {((col_nan_pct > 0) & (col_nan_pct < 5)).sum()}")
print(f"  Factors with 5-30% NaN: {((col_nan_pct >= 5) & (col_nan_pct < 30)).sum()}")
print(f"  Factors with 30-50% NaN: {((col_nan_pct >= 30) & (col_nan_pct < 50)).sum()}")
print(f"  Factors with ≥50% NaN: {(col_nan_pct >= 50).sum()}")

print(f"\n  {'Column':<35s} {'NaN %':>8s}  {'Count':>8s}  {'First Valid':>12s}  {'Last Valid':>12s}")
print("  " + "-" * 85)
for col, pct in col_nan_sorted.items():
    count = int(col_nan[col])
    valid = df[df[col].notna()]['date']
    first = valid.min().date() if len(valid) > 0 else 'N/A'
    last = valid.max().date() if len(valid) > 0 else 'N/A'
    flag = " ← DROP?" if pct >= 50 else (" ← INVESTIGATE" if pct >= 30 else "")
    print(f"  {col:<35s} {pct:>7.2f}%  {count:>8,d}  {str(first):>12s}  {str(last):>12s}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN:    {(row_nan == 0).sum():>6,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-5 NaN:  {((row_nan >= 1) & (row_nan <= 5)).sum():>6,d}")
print(f"  Rows with 6-15 NaN: {((row_nan > 5) & (row_nan <= 15)).sum():>6,d}")
print(f"  Rows with >15 NaN:  {(row_nan > 15).sum():>6,d}")
print(f"  Max NaN in any row: {row_nan.max()} out of {len(factor_cols)}")

# Show the worst rows (likely weekends/holidays)
worst_rows = df.loc[row_nan.nlargest(5).index, ['date']].copy()
worst_rows['n_nan'] = row_nan.nlargest(5).values
worst_rows['day_of_week'] = worst_rows['date'].dt.day_name()
print(f"\n  5 rows with most NaN:")
print(worst_rows.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: GAP ANALYSIS — Series-by-Series
# ═══════════════════════════════════════════════════════════════════════════════
#
# For macro data, NaN patterns come in three flavours:
#   1. Weekend/holiday gaps (scattered single-day NaN across all series)
#   2. Late-starting series (SOFR began 2018, IORB began 2021, etc.)
#   3. Discontinued series (LIBOR ended 2023, TED spread ended ~2022)
# We need to distinguish these because (1) gets forward-filled,
# (2) is structural and acceptable, (3) means we should truncate or drop.

# %%
print("\n" + "=" * 90)
print("STAGE 2: GAP ANALYSIS")
print("=" * 90)

# ── 2a. Date frequency check ────────────────────────────────────────────────
# Is this a trading-day calendar or a calendar-day file?
date_diffs = df['date'].diff().dt.days.dropna()
print(f"\n--- Date frequency ---")
print(f"  Most common gap: {date_diffs.mode().iloc[0]:.0f} day(s)")
print(f"  Gap distribution:")
for gap, count in date_diffs.value_counts().sort_index().head(10).items():
    label = {1: '1 day (consecutive)', 2: '2 days (over weekend?)',
             3: '3 days (weekend)', 4: '4 days (long weekend)'}.get(int(gap), f'{int(gap)} days')
    print(f"    {label}: {count:,} occurrences")

# ── 2b. Per-series coverage windows ─────────────────────────────────────────
print(f"\n--- Series coverage windows ---")
print(f"  {'Series':<35s} {'Start':>12s}  {'End':>12s}  {'Valid obs':>10s}  {'Status'}")
print("  " + "-" * 90)

for col in factor_cols:
    valid = df[df[col].notna()]['date']
    if len(valid) == 0:
        print(f"  {col:<35s} {'N/A':>12s}  {'N/A':>12s}  {'0':>10s}  EMPTY — DROP")
        continue

    first = valid.min().date()
    last = valid.max().date()
    n_valid = len(valid)

    # Classify the series
    if last < pd.Timestamp('2023-01-01').date():
        status = f"DISCONTINUED (ended {last})"
    elif first > pd.Timestamp('2010-01-01').date():
        status = f"LATE START (began {first})"
    elif n_valid < n_rows * 0.5:
        status = "SPARSE"
    else:
        status = "OK"

    print(f"  {col:<35s} {str(first):>12s}  {str(last):>12s}  {n_valid:>10,d}  {status}")

# ── 2c. Consecutive gap analysis for key series ─────────────────────────────
print(f"\n--- Longest consecutive NaN runs (trading-relevant series) ---")

key_series = ['yield_10y', 'yield_2y', 'fed_funds_eff', 'hy_oas', 'ig_oas',
              'wti_oil', 'gold', 'sp500', 'sofr', 'slope_2y10y']

for col in key_series:
    if col not in df.columns:
        continue
    series = df.set_index('date')[col]
    is_nan = series.isna()

    if not is_nan.any():
        print(f"  {col:<25s} No gaps ✓")
        continue

    # Find consecutive NaN runs
    runs = is_nan.ne(is_nan.shift()).cumsum()
    nan_groups = series[is_nan].groupby(runs).agg(['count', 'first', 'last'])

    longest = nan_groups['count'].max()
    total_gaps = int(is_nan.sum())

    if longest <= 3:
        print(f"  {col:<25s} {total_gaps} NaN total, longest run: {longest} days (safe to ffill)")
    else:
        print(f"  {col:<25s} {total_gaps} NaN total, longest run: {longest} days ← INVESTIGATE")
        # Show the top 3 longest gaps
        top_gaps = nan_groups.nlargest(3, 'count')
        for run_id, row in top_gaps.iterrows():
            gap_dates = series.index[runs == run_id]
            print(f"    Gap: {gap_dates[0].date()} → {gap_dates[-1].date()} "
                  f"({int(row['count'])} days)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: SPECIFIC ISSUE CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: SPECIFIC ISSUE CHECKS")
print("=" * 90)

# ── 3a. Duplicate FRED ID bug: ism_prices vs ppi_final ───────────────────────
# Both mapped to PPIFIS in the collection code. Check if they're identical.
print(f"\n--- Check: ism_prices vs ppi_final duplication ---")
if 'ism_prices' in df.columns and 'ppi_final' in df.columns:
    both_valid = df[['ism_prices', 'ppi_final']].dropna()
    if len(both_valid) > 0:
        are_equal = (both_valid['ism_prices'] == both_valid['ppi_final']).all()
        corr = both_valid['ism_prices'].corr(both_valid['ppi_final'])
        print(f"  Identical values: {are_equal}")
        print(f"  Correlation: {corr:.6f}")
        if are_equal or corr > 0.999:
            print(f"  → CONFIRMED DUPLICATE. Drop ism_prices (keep ppi_final).")
        else:
            print(f"  → NOT duplicate despite same FRED ID. Investigate.")
    else:
        print(f"  No overlapping valid observations to compare.")
elif 'ism_prices' not in df.columns:
    print(f"  ism_prices column not found (may not have been pulled).")
elif 'ppi_final' not in df.columns:
    print(f"  ppi_final column not found.")

# ── 3b. Discontinued series identification ───────────────────────────────────
print(f"\n--- Discontinued series (last valid date before 2024) ---")
discontinued = []
for col in factor_cols:
    valid = df[df[col].notna()]['date']
    if len(valid) > 0:
        last = valid.max()
        if last < pd.Timestamp('2024-01-01'):
            discontinued.append((col, last.date()))

if discontinued:
    for col, last_date in sorted(discontinued, key=lambda x: x[1]):
        print(f"  {col:<35s} last valid: {last_date}")
else:
    print(f"  None found — all series have data in 2024")

# ── 3c. Value range sanity checks ───────────────────────────────────────────
print(f"\n--- Value range sanity checks ---")
checks = {
    'yield_10y':     (0, 10,    '10Y yield should be 0-10%'),
    'yield_2y':      (-1, 10,   '2Y yield should be ~0-10%'),
    'fed_funds_eff':  (0, 10,    'Fed funds should be 0-10%'),
    'hy_oas':        (0, 30,    'HY OAS should be 0-30%'),
    'wti_oil':       (-50, 200, 'WTI should be -$50 to $200'),
    'gold':          (200, 3000, 'Gold should be $200-$3000'),
    'sp500':         (500, 7000, 'S&P 500 should be 500-7000'),
    'slope_2y10y':   (-3, 4,     '2s10s slope should be -3 to +4%'),
}

for col, (low, high, desc) in checks.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    n_out = ((vals < low) | (vals > high)).sum()
    if n_out > 0:
        print(f"  ⚠ {col:<25s} {n_out:>5d} values outside [{low}, {high}] — {desc}")
        outliers = vals[(vals < low) | (vals > high)]
        print(f"    Range: {outliers.min():.2f} to {outliers.max():.2f}")
    else:
        print(f"  ✓ {col:<25s} all values in [{low}, {high}]")

# ── 3d. Check for all-weekend rows ──────────────────────────────────────────
print(f"\n--- Weekend/holiday rows ---")
df['_dow'] = df['date'].dt.dayofweek  # 0=Mon, 6=Sun
weekend_rows = df[df['_dow'] >= 5]
print(f"  Saturday rows: {(df['_dow'] == 5).sum()}")
print(f"  Sunday rows:   {(df['_dow'] == 6).sum()}")
print(f"  Total weekend rows: {len(weekend_rows)}")
df = df.drop(columns='_dow')

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 4: SUMMARY — DECISIONS NEEDED BEFORE CLEANING")
print("=" * 90)

print(f"""
Review the output above and decide:

1. COLUMNS TO DROP (discontinued or duplicate):
   - ism_prices (if confirmed duplicate of ppi_final)
   - Any series with last valid date well before 2024
   - List them below before running the cleaning cell

2. ROWS TO DROP:
   - Weekend rows (if any found above)
   - Rows where ALL factors are NaN

3. FORWARD-FILL LIMIT:
   - Short gaps (1-3 days): forward-fill
   - Longer gaps: leave as NaN for now (will be handled in macro merge)

4. LATE-STARTING SERIES:
   - These are structural NaN, not data quality issues
   - Keep them — they'll simply be NaN before their start date
   - Document the start dates for the data dictionary

Once you've reviewed, paste back the output and I'll write the
cleaning/saving cell based on what the diagnostics show.
""")

STAGE 0: LOAD & INSPECT — fred_daily

Dropped 2,192 weekend rows (7,671 → 5,479)
Dropped 0 holiday rows (5,479 → 5,479)

After filtering:
  Shape: 5,479 rows × 69 columns
  Date range: 2004-01-01 → 2024-12-31

Shape: 5,479 rows × 69 columns
Date range: 2004-01-01 → 2024-12-31
Unique dates: 5,479

Factor columns (68):
    1. yield_1m                            float64        
    2. yield_3m                            float64        
    3. yield_6m                            float64        
    4. yield_1y                            float64        
    5. yield_2y                            float64        
    6. yield_3y                            float64        
    7. yield_5y                            float64        
    8. yield_7y                            float64        
    9. yield_10y                           float64        
   10. yield_20y                           float64        
   11. yield_30y                           float64        
   12. tips_5y                  

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 5: CLEAN & SAVE")
print("=" * 90)

# ── 5a. Drop columns ────────────────────────────────────────────────────────
drop_cols = [
    # ≥30% NaN
    'margin_debt', 'breakeven_20y', 'breakeven_30y', 'soma_mbs',
    'iorb', 'stlfsi', 'mortgage_30y', 'mortgage_15y', 'soma_treasury',
    'nfci', 'anfci', 'sofr', 'sp500', 'sp500_ret_1d', 'sp500_ret_5d',
    'rrp', 'tips_30y',
    # Discontinued (ends years before sample end)
    'ted_spread',
]

drop_cols_present = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols_present)

factor_cols = [c for c in df.columns if c != 'date']
print(f"\n  Dropped {len(drop_cols_present)} columns")
print(f"  Remaining: {len(factor_cols)} factor columns")

# ── 5b. Drop holiday rows ───────────────────────────────────────────────────
# After column drops, holidays show up as rows where almost everything is NaN.
# Drop rows where >50% of remaining factors are NaN.
nan_per_row = df[factor_cols].isna().sum(axis=1)
threshold = len(factor_cols) * 0.5
holiday_mask = nan_per_row > threshold

n_holidays = holiday_mask.sum()
df = df[~holiday_mask].reset_index(drop=True)
print(f"  Dropped {n_holidays} holiday rows")
print(f"  Remaining: {len(df):,} rows")

# ── 5c. Forward-fill remaining gaps (max 5 weekdays) ────────────────────────
nan_before = df[factor_cols].isna().sum().sum()
df = df.sort_values('date')
df[factor_cols] = df[factor_cols].ffill(limit=5)
nan_after_ffill = df[factor_cols].isna().sum().sum()
print(f"\n  Forward-fill (limit=5): {nan_before:,} → {nan_after_ffill:,} NaN")

# ── 5d. Final NaN report ────────────────────────────────────────────────────
# Remaining NaN should only be leading NaN for late-starting series
# (twexb/twexm from 2006, tips_20y from mid-2004). These are structural
# and will be resolved when the merged dataset starts from 2006.
nan_final = df[factor_cols].isna().sum()
nan_cols = nan_final[nan_final > 0]

if len(nan_cols) == 0:
    print(f"\n  ✓ Zero NaN — all clean")
else:
    print(f"\n  Remaining NaN ({len(nan_cols)} columns — expected late-start series only):")
    for col, n in nan_cols.sort_values(ascending=False).items():
        first_valid = df[df[col].notna()]['date'].min().date()
        print(f"    {col:<25s} {n:>5d} NaN  (first valid: {first_valid})")

# ── 5e. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Factor columns: {len(factor_cols)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols)} columns):")
for i, c in enumerate(factor_cols, 1):
    nan_n = df[c].isna().sum()
    nan_str = f"  ({nan_n} NaN)" if nan_n > 0 else ""
    print(f"    {i:>3d}. {c:<35s}{nan_str}")

print(f"\n  Sample (first 3 rows):")
print(df[['date'] + factor_cols[:8]].head(3).to_string(index=False))
print(f"\n  Sample (last 3 rows):")
print(df[['date'] + factor_cols[:8]].tail(3).to_string(index=False))

# ── 5f. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'fred_daily_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nCleaning complete.")

STAGE 5: CLEAN & SAVE

  Dropped 18 columns
  Remaining: 50 factor columns
  Dropped 225 holiday rows
  Remaining: 5,254 rows

  Forward-fill (limit=5): 1,469 → 1,149 NaN

  Remaining NaN (6 columns — expected late-start series only):
    twexb                       500 NaN  (first valid: 2006-01-03)
    twexm                       500 NaN  (first valid: 2006-01-03)
    tips_20y                    141 NaN  (first valid: 2004-07-27)
    natgas                        6 NaN  (first valid: 2004-01-05)
    wti_oil                       1 NaN  (first valid: 2004-01-05)
    brent_wti_spread              1 NaN  (first valid: 2004-01-05)

  Final shape: 5,254 rows × 51 columns
  Factor columns: 50
  Date range: 2004-01-02 → 2024-12-31

  Factor list (50 columns):
      1. yield_1m                           
      2. yield_3m                           
      3. yield_6m                           
      4. yield_1y                           
      5. yield_2y                           
      6. y

In [4]:
print(df[df['natgas'].isna()][['date']].to_string(index=False))

      date
2004-01-02
2005-09-30
2005-10-03
2005-10-04
2005-10-05
2005-10-06
